# ARIADNE — Phi-4-mini PRP-Graph · **100 queries × Top-20**

**Input:** Jina-v5 standalone 100Q Top-20  
**Method retained from the supplied 16-query notebook:** 10 rounds, bidirectional A/B and B/A next-token probabilities, PRP-Graph construction, graph-only weighted PageRank ranking.  
**Output:** 2,000 ranked rows + pairwise diagnostics + cache (on Drive).  
**GPU:** Google Colab, ideally L4/A100 with BF16. The first 1-query smoke test is recommended.  
**Required Drive files:** the canonical candidate Parquet and the chosen Top-20 CSV. No qrels needed to score.


## 1. Install dependencies

In [1]:
%pip install -q "transformers==4.53.3" "accelerate>=1.0,<2" safetensors sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 138.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 115.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Mount Google Drive and define paths

In [2]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SLM_RERANKING_100Q")
DATASET_PATH = PROJECT_ROOT / "data" / "candidate_profiles_english_clean_20k_with_text.parquet"
RETRIEVER_NAME = "jina_v5"
RANKING_PATH = PROJECT_ROOT / "dense_retrieval" / "jina_v5_100_queries_top20.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for path in (DATASET_PATH, RANKING_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Upload the required file to Google Drive first: {path}")
print("Project:",PROJECT_ROOT)
print("Dataset:",DATASET_PATH)
print("Ranking:",RANKING_PATH)
print("Output:",OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/SLM_RERANKING_100Q
Dataset: /content/drive/MyDrive/SLM_RERANKING_100Q/data/candidate_profiles_english_clean_20k_with_text.parquet
Ranking: /content/drive/MyDrive/SLM_RERANKING_100Q/dense_retrieval/jina_v5_100_queries_top20.csv
Output: /content/drive/MyDrive/SLM_RERANKING_100Q/outputs


## 3. Imports, GPU check and reproducibility

In [3]:

import gc
import hashlib
import inspect
import json
import logging
import math
import platform
import time
from dataclasses import asdict, dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python       :", platform.python_version())
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. In Colab choose Runtime → Change runtime type → GPU."
    )

print("GPU          :", torch.cuda.get_device_name(0))
print("BF16 support :", torch.cuda.is_bf16_supported())

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "This controlled experiment expects a BF16-capable GPU "
        "(for example A100/L4/H100)."
    )


Python       : 3.13.15
PyTorch      : 2.11.0+cu130
Transformers : 4.53.3
CUDA         : True
GPU          : NVIDIA A100-SXM4-40GB
BF16 support : True


## 4. Optional Hugging Face login

In [ ]:

# The model is public. Login is normally not required.
# If Hugging Face asks for authentication or you hit anonymous rate limits:
#
# from huggingface_hub import notebook_login
# notebook_login()


## 5. Experiment configuration

In [4]:
MODEL_NAME = "microsoft/Phi-4-mini-instruct"
MODEL_REVISION = None
MODEL_SHORT_NAME = "phi_4_mini_instruct"
PRECISION_MODE = "bf16"
PROMPT_VERSION = "recruitment_pairwise_v1"
RETRIEVER_NAME = "jina_v5"

TOP_K = 20
EXPECTED_QUERIES = 100
COMPARISON_ROUNDS = 10
MAX_CONTEXT_TOKENS = 15_360
CONTEXT_SAFETY_MARGIN = 128
FAIL_ON_TRUNCATION = False

PAGERANK_DAMPING = 0.85
PAGERANK_TOLERANCE = 1e-8
PAGERANK_MAX_ITERATIONS = 500
INTERPOLATION_ALPHA = 0.50
FINAL_SCORE_MODE = "graph_only"

RERANKER_NAME = f"{MODEL_SHORT_NAME}_full_top20_prp_graph_r{COMPARISON_ROUNDS}_ctx{MAX_CONTEXT_TOKENS}"
EXPERIMENT_DIR = OUTPUT_ROOT / RERANKER_NAME / RETRIEVER_NAME
LOG_DIR = OUTPUT_ROOT / RERANKER_NAME / "logs"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = 'You are an expert recruitment relevance evaluator.\n\nYour task is to compare two candidate profiles against one recruitment query and determine which candidate is more relevant and suitable for that query.\n\nUse only information explicitly stated in the recruitment query and candidate profiles. Do not invent, assume, or infer unstated skills, qualifications, experience, education, certifications, language proficiency, achievements, availability, or personal characteristics.\n\nApply the following job-related criteria only when they are relevant to the query:\n1. Direct match with the requested role, occupation, and professional domain.\n2. Coverage of explicitly requested technical, legal, business, or professional skills.\n3. Relevance, recency, and depth of demonstrated professional experience.\n4. Match with requested seniority and years of experience.\n5. Evidence that the requested skills were applied in actual work or projects.\n6. Relevant education, certifications, languages, sector knowledge, and responsibilities.\n7. Mandatory requirements must be prioritized over desirable requirements.\n8. Direct evidence must be preferred over vague semantic similarity or keyword repetition.\n\nDo not use or infer protected or sensitive characteristics, including age, sex, gender, nationality, ethnicity, religion, disability, health, marital or family status, photographs, names, or addresses. Do not prefer a candidate because the profile appears first, is longer, contains more repeated keywords, or uses a more polished writing style.\n\nYou must make a strict pairwise choice. Return exactly one label: A or B. Return no explanation and no other text.'

print("Model     :", MODEL_NAME)
print("Retriever :", RETRIEVER_NAME)
print("TOP_K     :", TOP_K)
print("Rounds    :", COMPARISON_ROUNDS)
print("Output    :", EXPERIMENT_DIR)

Model     : microsoft/Phi-4-mini-instruct
Retriever : jina_v5
TOP_K     : 20
Rounds    : 10
Output    : /content/drive/MyDrive/SLM_RERANKING_100Q/outputs/phi_4_mini_instruct_full_top20_prp_graph_r10_ctx15360/jina_v5


## 6. Load and validate the candidate dataset + raw Jina v5 standalone Top-20

In [5]:
REQUIRED_RANKING_COLUMNS = {"query_id", "category", "query", "candidate_id", "dataset_index", "rank"}

def load_candidate_dataset(path: Path) -> pd.DataFrame:
    """Use EXACT row-index identifiers, like the 100Q BGE/Nemotron/GTE rerankers."""
    df = pd.read_parquet(path).reset_index(drop=True)
    if "candidate_text" not in df.columns:
        raise ValueError("Dataset Parquet must contain candidate_text")
    df["candidate_id"] = df.index.astype(str)
    df["dataset_index"] = df.index.astype(int)
    df["candidate_text"] = df["candidate_text"].fillna("").astype(str)
    if df["candidate_text"].str.strip().eq("").any():
        raise ValueError("Empty candidate_text in original dataset")
    return df[["candidate_id", "dataset_index", "candidate_text"]].copy()

def load_retrieval_top20(path: Path, candidates: pd.DataFrame) -> pd.DataFrame:
    """Legacy function name retained; accepts the 100Q Top-20 ranking."""
    rankings = pd.read_csv(path, dtype={"query_id": str, "candidate_id": str})
    missing = REQUIRED_RANKING_COLUMNS - set(rankings.columns)
    if missing:
        raise ValueError(f"Missing source ranking columns: {sorted(missing)}")
    rankings["dataset_index"] = pd.to_numeric(rankings["dataset_index"], errors="raise").astype(int)
    rankings["rank"] = pd.to_numeric(rankings["rank"], errors="raise").astype(int)
    rankings = rankings.sort_values(["query_id", "rank"], kind="stable").reset_index(drop=True)
    if len(rankings) != EXPECTED_QUERIES * TOP_K or rankings["query_id"].nunique() != EXPECTED_QUERIES:
        raise ValueError(f"Expected {EXPECTED_QUERIES} queries × {TOP_K} candidates; got {rankings['query_id'].nunique()} queries / {len(rankings)} rows")
    if rankings.duplicated(["query_id", "candidate_id"]).any():
        raise ValueError("Duplicate query/candidate pairs")
    for qid, g in rankings.groupby("query_id", sort=False):
        if g["rank"].tolist() != list(range(1, TOP_K+1)):
            raise ValueError(f"{qid}: expected unique ranked candidates 1..{TOP_K}")
        if g["query"].nunique() != 1 or g["category"].nunique() != 1:
            raise ValueError(f"{qid}: inconsistent query/category text")
    # Preserve original retrieval confidence values for diagnostics; never mix
    # BGE/RRF scores across systems. Graph-only final score is the primary result.
    if "score" in rankings.columns:
        scores = pd.to_numeric(rankings["score"], errors="coerce")
    elif "rrf_score" in rankings.columns:
        scores = pd.to_numeric(rankings["rrf_score"], errors="coerce")
    else:
        scores = pd.Series(float("nan"), index=rankings.index)
    if scores.isna().any():
        print("NOTE: missing original retrieval scores were assigned 1/(60+rank) for diagnostics only")
        scores = scores.fillna(1.0/(60.0+rankings["rank"]))
    rankings["score"] = scores.astype(float)
    # Never trust possible CSV copies of candidate_text; rejoin with canonical Parquet.
    rankings = rankings.drop(columns=["candidate_text", "Position", "Moreinfo", "Looking For", "Highlights", "CV", "English Level", "Experience Years"], errors="ignore")
    merged = rankings.merge(candidates, on=["candidate_id", "dataset_index"], how="left", validate="many_to_one")
    if merged["candidate_text"].isna().any():
        examples = merged.loc[merged["candidate_text"].isna(), ["query_id", "candidate_id", "dataset_index"]].head(10)
        raise ValueError("Cannot map some candidate IDs to the original Parquet:\n" + examples.to_string(index=False))
    return merged

candidates = load_candidate_dataset(DATASET_PATH)
retrieval_top20 = load_retrieval_top20(RANKING_PATH, candidates) # legacy variable used throughout PRP-Graph notebook
print("DATA CHECK PASSED | candidates:", len(candidates), "| queries:", retrieval_top20.query_id.nunique(), "| ranking rows:", len(retrieval_top20))
display(retrieval_top20[["query_id", "candidate_id", "dataset_index", "rank", "score", "candidate_text"]].head(3))

DATA CHECK PASSED | candidates: 20000 | queries: 100 | ranking rows: 2000


,query_id,candidate_id,dataset_index,rank,score,candidate_text
0,Q001,14164,14164,1,0.691527,Position: CFO / Deputy CFO / Head of FP&A / Fi...
1,Q001,14331,14331,2,0.675110,Position: Chief Financial Officer | Head of Fi...
2,Q001,14182,14182,3,0.662867,"Position: CFO, Finance Manager, Finance Analys..."


## 7. Load Phi-4-mini-instruct in BF16

In [6]:
import transformers
print("Transformers:", transformers.__version__)

Transformers: 4.53.3


In [7]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

print("Loading Phi-4-mini-instruct...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

MODEL_CONTEXT_LIMIT = int(getattr(model.config, "max_position_embeddings", 0) or 0)
if MODEL_CONTEXT_LIMIT and MAX_CONTEXT_TOKENS > MODEL_CONTEXT_LIMIT:
    raise ValueError(f"Operational context {MAX_CONTEXT_TOKENS} exceeds model limit {MODEL_CONTEXT_LIMIT}.")

forward_parameters = inspect.signature(model.forward).parameters
if "logits_to_keep" not in forward_parameters:
    raise RuntimeError(
        "This Phi/Transformers build does not expose logits_to_keep. "
        "Re-run the install cell, restart the Colab session, then run from the top."
    )

def model_input_device(model: torch.nn.Module) -> torch.device:
    embeddings = model.get_input_embeddings()
    if embeddings is None or not hasattr(embeddings, "weight"):
        raise RuntimeError("Could not locate Phi input embeddings.")
    return embeddings.weight.device

print("Model context limit :", MODEL_CONTEXT_LIMIT)
print("Operational context :", MAX_CONTEXT_TOKENS)
print("Input device        :", model_input_device(model))
print("Device map          :", getattr(model, "hf_device_map", None))

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Loading Phi-4-mini-instruct...


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Model context limit : 131072
Operational context : 15360
Input device        : cuda:0
Device map          : {'': 0}


## 8. Pairwise prompt preparation and exact A/B probability scoring

In [8]:

@dataclass(frozen=True)
class PreparedPrompt:
    input_ids: torch.Tensor
    prompt_tokens: int
    candidate_a_original_tokens: int
    candidate_b_original_tokens: int
    candidate_a_used_tokens: int
    candidate_b_used_tokens: int
    truncated: bool


@dataclass(frozen=True)
class PairwiseResult:
    query_id: str
    candidate_first: str
    candidate_second: str
    dataset_index_first: int
    dataset_index_second: int

    p_first_when_first: float
    p_second_when_first: float

    forward_p_a: float
    forward_p_b: float
    reverse_p_a: float
    reverse_p_b: float

    forward_prompt_tokens: int
    reverse_prompt_tokens: int

    first_original_tokens: int
    second_original_tokens: int

    first_used_tokens_forward: int
    second_used_tokens_forward: int
    second_used_tokens_reverse: int
    first_used_tokens_reverse: int

    forward_truncated: bool
    reverse_truncated: bool

    forward_winner: str
    reverse_winner: str
    order_consistent: bool


def as_id_list(ids) -> list[int]:
    if isinstance(ids, torch.Tensor):
        ids = ids.detach().cpu().tolist()
    if isinstance(ids, int):
        ids = [ids]
    return [int(x) for x in ids]


def get_single_token_label_ids(tokenizer) -> dict[str, int]:
    encoded = {
        label: as_id_list(
            tokenizer.encode(label, add_special_tokens=False)
        )
        for label in ("A", "B")
    }

    invalid = {
        label: ids
        for label, ids in encoded.items()
        if len(ids) != 1
    }
    if invalid:
        raise RuntimeError(
            "Output labels A/B must each tokenize as exactly one token. "
            f"Observed: {encoded}"
        )

    if encoded["A"][0] == encoded["B"][0]:
        raise RuntimeError("A and B map to the same token ID.")

    return {
        "A": encoded["A"][0],
        "B": encoded["B"][0],
    }


LABEL_TOKEN_IDS = get_single_token_label_ids(tokenizer)
print("A/B token IDs:", LABEL_TOKEN_IDS)


def build_user_prompt(
    query: str,
    candidate_a: str,
    candidate_b: str,
) -> str:
    return f"""Recruitment query:
<BEGIN_QUERY>
{query.strip()}
<END_QUERY>

Candidate A:
<BEGIN_CANDIDATE_A>
{candidate_a.strip()}
<END_CANDIDATE_A>

Candidate B:
<BEGIN_CANDIDATE_B>
{candidate_b.strip()}
<END_CANDIDATE_B>

Which candidate is more relevant and suitable for the recruitment query?
Return exactly one label: A or B."""


def render_prompt_ids(
    tokenizer,
    query: str,
    candidate_a: str,
    candidate_b: str,
) -> torch.Tensor:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": build_user_prompt(
                query=query,
                candidate_a=candidate_a,
                candidate_b=candidate_b,
            ),
        },
    ]

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    if isinstance(rendered, dict):
        input_ids = rendered["input_ids"]
    elif hasattr(rendered, "input_ids"):
        input_ids = rendered.input_ids
    elif isinstance(rendered, torch.Tensor):
        input_ids = rendered
    else:
        input_ids = torch.tensor(rendered, dtype=torch.long)

    if not isinstance(input_ids, torch.Tensor):
        input_ids = torch.tensor(input_ids, dtype=torch.long)

    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    return input_ids.cpu()


def allocate_candidate_budgets(
    length_a: int,
    length_b: int,
    available: int,
) -> tuple[int, int]:
    if available <= 0:
        raise ValueError(
            "System/query/template tokens alone exceed the context budget."
        )

    if length_a + length_b <= available:
        return length_a, length_b

    budget_a = min(length_a, available // 2)
    budget_b = min(length_b, available - budget_a)

    remaining = available - budget_a - budget_b

    if remaining > 0:
        extra_a = min(length_a - budget_a, remaining)
        budget_a += extra_a
        remaining -= extra_a

    if remaining > 0:
        extra_b = min(length_b - budget_b, remaining)
        budget_b += extra_b

    return max(1, budget_a), max(1, budget_b)


def prepare_pair_prompt(
    tokenizer,
    query: str,
    candidate_a: str,
    candidate_b: str,
) -> PreparedPrompt:
    candidate_a_ids = as_id_list(
        tokenizer.encode(candidate_a, add_special_tokens=False)
    )
    candidate_b_ids = as_id_list(
        tokenizer.encode(candidate_b, add_special_tokens=False)
    )

    original_a = len(candidate_a_ids)
    original_b = len(candidate_b_ids)

    full_prompt_ids = render_prompt_ids(
        tokenizer=tokenizer,
        query=query,
        candidate_a=candidate_a,
        candidate_b=candidate_b,
    )

    usable_limit = MAX_CONTEXT_TOKENS - CONTEXT_SAFETY_MARGIN

    if full_prompt_ids.shape[1] <= usable_limit:
        return PreparedPrompt(
            input_ids=full_prompt_ids,
            prompt_tokens=int(full_prompt_ids.shape[1]),
            candidate_a_original_tokens=original_a,
            candidate_b_original_tokens=original_b,
            candidate_a_used_tokens=original_a,
            candidate_b_used_tokens=original_b,
            truncated=False,
        )

    if FAIL_ON_TRUNCATION:
        raise ValueError(
            "Pairwise prompt exceeds configured context budget: "
            f"{full_prompt_ids.shape[1]} > {usable_limit}"
        )

    empty_prompt_ids = render_prompt_ids(
        tokenizer=tokenizer,
        query=query,
        candidate_a="",
        candidate_b="",
    )

    fixed_tokens = int(empty_prompt_ids.shape[1])
    candidate_budget = usable_limit - fixed_tokens

    budget_a, budget_b = allocate_candidate_budgets(
        length_a=original_a,
        length_b=original_b,
        available=candidate_budget,
    )

    used_a_ids = candidate_a_ids[:budget_a]
    used_b_ids = candidate_b_ids[:budget_b]

    while True:
        fitted_a = tokenizer.decode(
            used_a_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        fitted_b = tokenizer.decode(
            used_b_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        fitted_prompt_ids = render_prompt_ids(
            tokenizer=tokenizer,
            query=query,
            candidate_a=fitted_a,
            candidate_b=fitted_b,
        )

        prompt_length = int(fitted_prompt_ids.shape[1])
        if prompt_length <= usable_limit:
            break

        excess = prompt_length - usable_limit
        reduction = max(8, math.ceil(excess / 2))

        if len(used_a_ids) >= len(used_b_ids) and len(used_a_ids) > 1:
            used_a_ids = used_a_ids[
                : max(1, len(used_a_ids) - reduction)
            ]
        elif len(used_b_ids) > 1:
            used_b_ids = used_b_ids[
                : max(1, len(used_b_ids) - reduction)
            ]
        else:
            raise RuntimeError(
                "Could not fit the pairwise prompt into the context budget."
            )

    return PreparedPrompt(
        input_ids=fitted_prompt_ids,
        prompt_tokens=int(fitted_prompt_ids.shape[1]),
        candidate_a_original_tokens=original_a,
        candidate_b_original_tokens=original_b,
        candidate_a_used_tokens=len(used_a_ids),
        candidate_b_used_tokens=len(used_b_ids),
        truncated=True,
    )


@torch.inference_mode()
def score_binary_labels(
    model: torch.nn.Module,
    prompt_ids: torch.Tensor,
) -> dict[str, float]:
    """
    Compute full-vocabulary next-token probabilities, then extract A and B.
    The A/B probabilities are NOT renormalized to sum to 1.
    """
    device = model_input_device(model)

    input_ids = prompt_ids.to(device, non_blocking=True)
    attention_mask = torch.ones_like(input_ids)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        logits_to_keep=1,
    )

    next_token_logits = outputs.logits[0, -1].float()
    full_vocab_probs = torch.softmax(
        next_token_logits,
        dim=-1,
    )

    return {
        "A": float(
            full_vocab_probs[LABEL_TOKEN_IDS["A"]].detach().cpu()
        ),
        "B": float(
            full_vocab_probs[LABEL_TOKEN_IDS["B"]].detach().cpu()
        ),
    }


A/B token IDs: {'A': 32, 'B': 33}


## 9. Persistent JSONL cache + bidirectional PRP unit

In [9]:

class PairwiseCache:
    """Persistent Drive cache for safe Colab resume."""

    def __init__(self, path: Path) -> None:
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._items: dict[tuple[str, str, str], PairwiseResult] = {}
        self._load()

    @staticmethod
    def _key(
        query_id: str,
        candidate_a: str,
        candidate_b: str,
    ) -> tuple[str, str, str]:
        first, second = sorted(
            [str(candidate_a), str(candidate_b)]
        )
        return str(query_id), first, second

    def _load(self) -> None:
        if not self.path.exists():
            return

        with self.path.open("r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    result = PairwiseResult(**json.loads(line))
                except Exception as error:
                    raise ValueError(
                        f"Invalid cache line {line_number}: {error}"
                    ) from error

                key = self._key(
                    result.query_id,
                    result.candidate_first,
                    result.candidate_second,
                )
                self._items[key] = result

        print(
            f"Loaded {len(self._items)} cached pairwise comparisons."
        )

    def get(
        self,
        query_id: str,
        candidate_a: str,
        candidate_b: str,
    ) -> PairwiseResult | None:
        return self._items.get(
            self._key(
                query_id,
                candidate_a,
                candidate_b,
            )
        )

    def append(self, result: PairwiseResult) -> None:
        key = self._key(
            result.query_id,
            result.candidate_first,
            result.candidate_second,
        )
        self._items[key] = result

        with self.path.open("a", encoding="utf-8") as file:
            file.write(
                json.dumps(
                    asdict(result),
                    ensure_ascii=False,
                )
                + "\n"
            )
            file.flush()


def compare_pair_bidirectionally(
    *,
    model: torch.nn.Module,
    tokenizer,
    cache: PairwiseCache,
    query_id: str,
    query: str,
    candidate_i: pd.Series,
    candidate_j: pd.Series,
) -> PairwiseResult:
    """
    Run u(q, di, dj) and u(q, dj, di), or reuse the cached result.
    """
    candidate_i_id = str(candidate_i["candidate_id"])
    candidate_j_id = str(candidate_j["candidate_id"])

    cached = cache.get(
        query_id,
        candidate_i_id,
        candidate_j_id,
    )
    if cached is not None:
        return cached

    text_i = str(candidate_i["candidate_text"])
    text_j = str(candidate_j["candidate_text"])

    # Forward order: i=A, j=B
    forward = prepare_pair_prompt(
        tokenizer=tokenizer,
        query=query,
        candidate_a=text_i,
        candidate_b=text_j,
    )
    forward_probabilities = score_binary_labels(
        model=model,
        prompt_ids=forward.input_ids,
    )

    # Reverse order: j=A, i=B
    reverse = prepare_pair_prompt(
        tokenizer=tokenizer,
        query=query,
        candidate_a=text_j,
        candidate_b=text_i,
    )
    reverse_probabilities = score_binary_labels(
        model=model,
        prompt_ids=reverse.input_ids,
    )

    p_i_when_first = float(forward_probabilities["A"])
    p_j_when_first = float(reverse_probabilities["A"])

    forward_winner = (
        candidate_i_id
        if forward_probabilities["A"] >= forward_probabilities["B"]
        else candidate_j_id
    )
    reverse_winner = (
        candidate_j_id
        if reverse_probabilities["A"] >= reverse_probabilities["B"]
        else candidate_i_id
    )

    result = PairwiseResult(
        query_id=str(query_id),
        candidate_first=candidate_i_id,
        candidate_second=candidate_j_id,
        dataset_index_first=int(candidate_i["dataset_index"]),
        dataset_index_second=int(candidate_j["dataset_index"]),
        p_first_when_first=p_i_when_first,
        p_second_when_first=p_j_when_first,
        forward_p_a=float(forward_probabilities["A"]),
        forward_p_b=float(forward_probabilities["B"]),
        reverse_p_a=float(reverse_probabilities["A"]),
        reverse_p_b=float(reverse_probabilities["B"]),
        forward_prompt_tokens=forward.prompt_tokens,
        reverse_prompt_tokens=reverse.prompt_tokens,
        first_original_tokens=forward.candidate_a_original_tokens,
        second_original_tokens=forward.candidate_b_original_tokens,
        first_used_tokens_forward=forward.candidate_a_used_tokens,
        second_used_tokens_forward=forward.candidate_b_used_tokens,
        second_used_tokens_reverse=reverse.candidate_a_used_tokens,
        first_used_tokens_reverse=reverse.candidate_b_used_tokens,
        forward_truncated=forward.truncated,
        reverse_truncated=reverse.truncated,
        forward_winner=forward_winner,
        reverse_winner=reverse_winner,
        order_consistent=(forward_winner == reverse_winner),
    )

    cache.append(result)
    return result


def probabilities_for_requested_order(
    result: PairwiseResult,
    candidate_i_id: str,
    candidate_j_id: str,
) -> tuple[float, float]:
    candidate_i_id = str(candidate_i_id)
    candidate_j_id = str(candidate_j_id)

    if (
        result.candidate_first == candidate_i_id
        and result.candidate_second == candidate_j_id
    ):
        return (
            result.p_first_when_first,
            result.p_second_when_first,
        )

    if (
        result.candidate_first == candidate_j_id
        and result.candidate_second == candidate_i_id
    ):
        return (
            result.p_second_when_first,
            result.p_first_when_first,
        )

    raise ValueError(
        "Cached pair does not match the requested candidates."
    )


## 10. Full Top-20 PRP-Graph construction

In [10]:

def build_prp_graph_for_query(
    *,
    group: pd.DataFrame,
    model: torch.nn.Module,
    tokenizer,
    cache: PairwiseCache,
    logger: logging.Logger,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, np.ndarray]:
    """
    Construct the pairwise ranking graph over ALL 20 candidates.

    Edge convention:
        W[j, i] = probability supporting candidate i over candidate j
        W[i, j] = probability supporting candidate j over candidate i
    """
    group = (
        group.sort_values("rank", kind="stable")
        .reset_index(drop=True)
        .copy()
    )

    query_id = str(group["query_id"].iloc[0])
    query = str(group["query"].iloc[0])

    candidate_ids = group["candidate_id"].astype(str).tolist()
    number_of_candidates = len(candidate_ids)

    if number_of_candidates != TOP_K:
        raise ValueError(
            f"Query {query_id} has {number_of_candidates} candidates; "
            f"expected {TOP_K}."
        )

    rows_by_candidate = {
        str(row["candidate_id"]): row
        for _, row in group.iterrows()
    }
    initial_rank = {
        str(row["candidate_id"]): int(row["rank"])
        for _, row in group.iterrows()
    }

    # Same initialization as the large-Mistral PRP-Graph experiment:
    # S0 = [1, 1-1/N, 1-2/N, ..., 1/N]
    construction_scores = {
        candidate_id: 1.0 - (
            position / number_of_candidates
        )
        for position, candidate_id in enumerate(candidate_ids)
    }

    current_order = candidate_ids.copy()
    compared_pairs: set[frozenset[str]] = set()

    id_to_matrix_index = {
        candidate_id: index
        for index, candidate_id in enumerate(candidate_ids)
    }

    edge_weights = np.zeros(
        (number_of_candidates, number_of_candidates),
        dtype=np.float64,
    )

    pair_records: list[dict[str, Any]] = []
    round_records: list[dict[str, Any]] = []

    rounds_completed = 0

    for round_number in range(
        1,
        COMPARISON_ROUNDS + 1,
    ):
        used_this_round: set[str] = set()
        previous_scores = construction_scores.copy()
        updated_scores = previous_scores.copy()
        comparisons_this_round = 0

        for position_i, candidate_i_id in enumerate(
            current_order
        ):
            if candidate_i_id in used_this_round:
                continue

            candidate_j_id = None

            for candidate_j_id_attempt in current_order[
                position_i + 1 :
            ]:
                pair_key = frozenset(
                    [
                        candidate_i_id,
                        candidate_j_id_attempt,
                    ]
                )

                if candidate_j_id_attempt in used_this_round:
                    continue
                if pair_key in compared_pairs:
                    continue

                candidate_j_id = candidate_j_id_attempt
                break

            if candidate_j_id is None:
                continue

            candidate_i = rows_by_candidate[candidate_i_id]
            candidate_j = rows_by_candidate[candidate_j_id]

            result = compare_pair_bidirectionally(
                model=model,
                tokenizer=tokenizer,
                cache=cache,
                query_id=query_id,
                query=query,
                candidate_i=candidate_i,
                candidate_j=candidate_j,
            )

            p_i, p_j = probabilities_for_requested_order(
                result=result,
                candidate_i_id=candidate_i_id,
                candidate_j_id=candidate_j_id,
            )

            matrix_i = id_to_matrix_index[candidate_i_id]
            matrix_j = id_to_matrix_index[candidate_j_id]

            # Directed edge j -> i expresses evidence supporting i over j.
            edge_weights[matrix_j, matrix_i] = p_i

            # Directed edge i -> j expresses evidence supporting j over i.
            edge_weights[matrix_i, matrix_j] = p_j

            updated_scores[candidate_i_id] = (
                previous_scores[candidate_i_id]
                + p_i
                * previous_scores[candidate_j_id]
                / round_number
            )
            updated_scores[candidate_j_id] = (
                previous_scores[candidate_j_id]
                + p_j
                * previous_scores[candidate_i_id]
                / round_number
            )

            used_this_round.update(
                [candidate_i_id, candidate_j_id]
            )
            compared_pairs.add(
                frozenset(
                    [candidate_i_id, candidate_j_id]
                )
            )
            comparisons_this_round += 1

            pair_records.append(
                {
                    **asdict(result),
                    "round": round_number,
                    "candidate_i": candidate_i_id,
                    "candidate_j": candidate_j_id,
                    "p_i_graph_edge": p_i,
                    "p_j_graph_edge": p_j,
                    "i_rank_before_round": (
                        current_order.index(candidate_i_id) + 1
                    ),
                    "j_rank_before_round": (
                        current_order.index(candidate_j_id) + 1
                    ),
                    "i_construction_score_before": (
                        previous_scores[candidate_i_id]
                    ),
                    "j_construction_score_before": (
                        previous_scores[candidate_j_id]
                    ),
                }
            )

        if comparisons_this_round == 0:
            logger.warning(
                "Query %s stopped after %d completed rounds "
                "because no unseen pairs remained.",
                query_id,
                rounds_completed,
            )
            break

        construction_scores = updated_scores

        current_order = sorted(
            current_order,
            key=lambda candidate_id: (
                -construction_scores[candidate_id],
                initial_rank[candidate_id],
            ),
        )

        rounds_completed = round_number

        for temporary_rank, candidate_id in enumerate(
            current_order,
            start=1,
        ):
            round_records.append(
                {
                    "query_id": query_id,
                    "round": round_number,
                    "candidate_id": candidate_id,
                    "temporary_rank": temporary_rank,
                    "construction_score": (
                        construction_scores[candidate_id]
                    ),
                    "comparisons_this_round": (
                        comparisons_this_round
                    ),
                    "cumulative_unique_pairs": (
                        len(compared_pairs)
                    ),
                }
            )

        logger.info(
            "Query %s | round %d/%d | pairs=%d | cumulative=%d",
            query_id,
            round_number,
            COMPARISON_ROUNDS,
            comparisons_this_round,
            len(compared_pairs),
        )

    construction_df = group.copy()

    construction_df["construction_score"] = (
        construction_df["candidate_id"]
        .astype(str)
        .map(construction_scores)
    )
    construction_df["rounds_completed"] = (
        rounds_completed
    )
    construction_df["unique_pairs_compared"] = (
        len(compared_pairs)
    )

    pair_df = pd.DataFrame(pair_records)
    round_df = pd.DataFrame(round_records)

    return (
        construction_df,
        pair_df,
        round_df,
        edge_weights,
    )


## 11. Weighted PageRank aggregation and final rank 1–50

In [11]:

def weighted_pagerank(
    edge_weights: np.ndarray,
    initial_distribution: np.ndarray,
) -> tuple[np.ndarray, int, bool]:
    """Compute weighted PageRank over the PRP comparison graph."""

    if (
        edge_weights.ndim != 2
        or edge_weights.shape[0] != edge_weights.shape[1]
    ):
        raise ValueError(
            "edge_weights must be a square matrix."
        )

    number_of_nodes = edge_weights.shape[0]

    initial = np.asarray(
        initial_distribution,
        dtype=np.float64,
    )
    initial = np.clip(
        initial,
        a_min=0.0,
        a_max=None,
    )

    if initial.sum() <= 0:
        initial = np.full(
            number_of_nodes,
            1.0 / number_of_nodes,
        )
    else:
        initial = initial / initial.sum()

    scores = initial.copy()

    out_weight_sums = edge_weights.sum(axis=1)
    uniform_teleport = np.full(
        number_of_nodes,
        1.0 / number_of_nodes,
    )

    converged = False

    for iteration in range(
        1,
        PAGERANK_MAX_ITERATIONS + 1,
    ):
        new_scores = (
            (1.0 - PAGERANK_DAMPING)
            * uniform_teleport
        )

        dangling_mass = float(
            scores[out_weight_sums == 0.0].sum()
        )

        if dangling_mass > 0:
            new_scores += (
                PAGERANK_DAMPING
                * dangling_mass
                * uniform_teleport
            )

        for source in range(number_of_nodes):
            out_sum = out_weight_sums[source]

            if out_sum <= 0:
                continue

            new_scores += (
                PAGERANK_DAMPING
                * scores[source]
                * edge_weights[source]
                / out_sum
            )

        new_scores_sum = float(new_scores.sum())

        if new_scores_sum <= 0:
            raise RuntimeError(
                "PageRank produced a non-positive score sum."
            )

        new_scores /= new_scores_sum

        if (
            np.linalg.norm(
                new_scores - scores,
                ord=1,
            )
            < PAGERANK_TOLERANCE
        ):
            scores = new_scores
            converged = True
            return scores, iteration, converged

        scores = new_scores

    return (
        scores,
        PAGERANK_MAX_ITERATIONS,
        converged,
    )


def aggregate_and_rank_query(
    construction_df: pd.DataFrame,
    edge_weights: np.ndarray,
) -> pd.DataFrame:
    """
    Produce a complete NEW ranking 1..20.
    """

    group = (
        construction_df
        .sort_values("rank", kind="stable")
        .reset_index(drop=True)
        .copy()
    )

    initial_scores = group[
        "score"
    ].to_numpy(dtype=np.float64)

    initial_scores = np.clip(
        initial_scores,
        a_min=0.0,
        a_max=None,
    )

    if initial_scores.sum() <= 0:
        initial_normalized = np.full(
            len(group),
            1.0 / len(group),
            dtype=np.float64,
        )
    else:
        initial_normalized = (
            initial_scores
            / initial_scores.sum()
        )

    graph_scores, iterations, converged = (
        weighted_pagerank(
            edge_weights=edge_weights,
            initial_distribution=initial_normalized,
        )
    )

    interpolated_scores = (
        INTERPOLATION_ALPHA * graph_scores
        + (
            1.0 - INTERPOLATION_ALPHA
        ) * initial_normalized
    )

    group["initial_rank"] = group[
        "rank"
    ].astype(int)
    group["initial_rrf_score"] = group[
        "score"
    ].astype(float)
    group["initial_rrf_normalized"] = (
        initial_normalized
    )

    group["prp_graph_score"] = graph_scores
    group[
        "prp_graph_rrf_interpolated_score"
    ] = interpolated_scores

    group["pagerank_iterations"] = iterations
    group["pagerank_converged"] = converged

    if FINAL_SCORE_MODE == "graph_only":
        group["final_score"] = group[
            "prp_graph_score"
        ]
    elif FINAL_SCORE_MODE == "interpolated":
        group["final_score"] = group[
            "prp_graph_rrf_interpolated_score"
        ]
    else:
        raise ValueError(
            "FINAL_SCORE_MODE must be "
            "'graph_only' or 'interpolated'."
        )

    group = group.sort_values(
        ["final_score", "initial_rank"],
        ascending=[False, True],
        kind="stable",
    ).reset_index(drop=True)

    group["reranked_rank"] = np.arange(
        1,
        len(group) + 1,
    )

    group["rank_change"] = (
        group["initial_rank"]
        - group["reranked_rank"]
    )

    group["model"] = MODEL_NAME
    group["reranker"] = "PRP-Graph"
    group["final_score_mode"] = (
        FINAL_SCORE_MODE
    )

    # Compact evaluation columns use the NEW ranking.
    group["rank"] = group[
        "reranked_rank"
    ].astype(int)
    group["score"] = group[
        "final_score"
    ].astype(float)

    return group


## 12. Experiment signature, logger and full-system runner

In [12]:

def experiment_signature() -> str:
    payload = {
        "model": MODEL_NAME,
        "model_revision": MODEL_REVISION,
        "precision_mode": PRECISION_MODE,
        "top_k": TOP_K,
        "comparison_rounds": COMPARISON_ROUNDS,
        "max_context_tokens": MAX_CONTEXT_TOKENS,
        "context_safety_margin": CONTEXT_SAFETY_MARGIN,
        "prompt_version": PROMPT_VERSION,
        "system_prompt": SYSTEM_PROMPT,
        "final_score_mode": FINAL_SCORE_MODE,
        "seed": SEED,
    }

    serialized = json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")

    return hashlib.sha256(
        serialized
    ).hexdigest()[:12]


def create_logger(
    log_path: Path,
) -> logging.Logger:
    logger = logging.getLogger(
        log_path.stem
    )
    logger.setLevel(logging.INFO)
    logger.propagate = False

    if logger.handlers:
        logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(
        log_path,
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)

    logger.addHandler(stream_handler)
    logger.addHandler(file_handler)

    return logger


def rerank_system(
    ranking: pd.DataFrame,
    *,
    max_queries: int | None = None,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
]:
    query_ids = (
        ranking["query_id"]
        .drop_duplicates()
        .tolist()
    )

    if max_queries is not None:
        query_ids = query_ids[
            :max_queries
        ]

    cache_path = (
        EXPERIMENT_DIR
        / f"cache_{experiment_signature()}.jsonl"
    )

    cache = PairwiseCache(
        cache_path
    )

    log_path = (
        LOG_DIR
        / (
            f"{RERANKER_NAME}"
            "__hybrid_bm25_jina_rrf.log"
        )
    )

    logger = create_logger(
        log_path
    )

    reranked_groups = []
    all_pair_records = []
    all_round_records = []

    for query_id in tqdm(
        query_ids,
        desc="Ministral-3-3B full Top-20 PRP-Graph",
    ):
        group = (
            ranking.loc[
                ranking["query_id"]
                == query_id
            ]
            .sort_values(
                "rank",
                kind="stable",
            )
            .reset_index(drop=True)
            .copy()
        )

        logger.info(
            "Starting query %s with %d candidates.",
            query_id,
            len(group),
        )

        (
            construction_df,
            pair_df,
            round_df,
            edge_weights,
        ) = build_prp_graph_for_query(
            group=group,
            model=model,
            tokenizer=tokenizer,
            cache=cache,
            logger=logger,
        )

        reranked = aggregate_and_rank_query(
            construction_df=construction_df,
            edge_weights=edge_weights,
        )

        reranked["retriever"] = (
            RETRIEVER_NAME
        )

        reranked_groups.append(
            reranked
        )

        if not pair_df.empty:
            pair_df["retriever"] = (
                RETRIEVER_NAME
            )
            all_pair_records.append(
                pair_df
            )

        if not round_df.empty:
            round_df["retriever"] = (
                RETRIEVER_NAME
            )
            all_round_records.append(
                round_df
            )

        logger.info(
            "Completed query %s | pairs=%d | top=%s",
            query_id,
            int(
                reranked[
                    "unique_pairs_compared"
                ].iloc[0]
            ),
            str(
                reranked[
                    "candidate_id"
                ].iloc[0]
            ),
        )

        gc.collect()
        torch.cuda.empty_cache()

    if not reranked_groups:
        raise RuntimeError(
            "No query groups were reranked."
        )

    reranked_all = pd.concat(
        reranked_groups,
        ignore_index=True,
    )

    reranked_all = reranked_all.sort_values(
        ["query_id", "rank"],
        kind="stable",
    ).reset_index(drop=True)

    pairwise_all = (
        pd.concat(
            all_pair_records,
            ignore_index=True,
        )
        if all_pair_records
        else pd.DataFrame()
    )

    rounds_all = (
        pd.concat(
            all_round_records,
            ignore_index=True,
        )
        if all_round_records
        else pd.DataFrame()
    )

    return (
        reranked_all,
        pairwise_all,
        rounds_all,
    )


## 13. Validate the FULL Top-20 reranking

In [13]:

def validate_full_top20(
    original: pd.DataFrame,
    reranked: pd.DataFrame,
) -> None:
    original_query_ids = (
        original["query_id"]
        .drop_duplicates()
        .tolist()
    )
    reranked_query_ids = (
        reranked["query_id"]
        .drop_duplicates()
        .tolist()
    )

    if original_query_ids != reranked_query_ids:
        raise AssertionError(
            "Reranked query IDs differ from the original input."
        )

    changed_order_queries = 0

    for query_id in original_query_ids:
        original_group = (
            original[
                original["query_id"]
                == query_id
            ]
            .sort_values("rank")
            .reset_index(drop=True)
        )

        reranked_group = (
            reranked[
                reranked["query_id"]
                == query_id
            ]
            .sort_values("rank")
            .reset_index(drop=True)
        )

        if len(reranked_group) != TOP_K:
            raise AssertionError(
                f"{query_id}: expected 50 reranked candidates."
            )

        if reranked_group["rank"].tolist() != list(
            range(1, TOP_K + 1)
        ):
            raise AssertionError(
                f"{query_id}: final ranks are not exactly 1..20."
            )

        original_set = set(
            original_group[
                "candidate_id"
            ].astype(str)
        )
        reranked_set = set(
            reranked_group[
                "candidate_id"
            ].astype(str)
        )

        if original_set != reranked_set:
            raise AssertionError(
                f"{query_id}: the Top-20 candidate SET changed."
            )

        original_order = (
            original_group[
                "candidate_id"
            ]
            .astype(str)
            .tolist()
        )
        reranked_order = (
            reranked_group[
                "candidate_id"
            ]
            .astype(str)
            .tolist()
        )

        if original_order != reranked_order:
            changed_order_queries += 1

        if not np.all(
            np.diff(
                reranked_group[
                    "score"
                ].to_numpy(float)
            )
            <= 1e-15
        ):
            raise AssertionError(
                f"{query_id}: final scores are not non-increasing by rank."
            )

    print("VALIDATION PASSED")
    print("✓ every query has exactly 20 candidates")
    print("✓ final ranks are exactly 1..20")
    print("✓ the same original Top-20 candidate set is preserved")
    print("✓ ALL 20 candidates were eligible to move")
    print(
        "✓ queries whose final order changed:",
        changed_order_queries,
        "/",
        len(original_query_ids),
    )


## 14. Save rankings, diagnostics, cache and configuration

In [14]:
def create_evaluation_ranking(reranked: pd.DataFrame) -> pd.DataFrame:
    evaluation = reranked.copy()
    evaluation["system"] = f"Rerank-{RETRIEVER_NAME}__{MODEL_SHORT_NAME}"
    return evaluation[["system", "query_id", "category", "query", "candidate_id", "dataset_index", "rank", "score"]].copy()

def save_run_configuration(output_path: Path) -> None:
    payload = dict(
        model=MODEL_NAME, model_revision=MODEL_REVISION, precision_mode=PRECISION_MODE,
        prompt_version=PROMPT_VERSION, method="PRP-Graph", initial_retriever=RETRIEVER_NAME,
        candidate_pool=f"{EXPECTED_QUERIES} queries / Top-{TOP_K}",
        full_reranking_depth=TOP_K, comparison_rounds=COMPARISON_ROUNDS,
        max_context_tokens=MAX_CONTEXT_TOKENS, context_safety_margin=CONTEXT_SAFETY_MARGIN,
        fail_on_truncation=FAIL_ON_TRUNCATION, pagerank_damping=PAGERANK_DAMPING,
        pagerank_tolerance=PAGERANK_TOLERANCE, pagerank_max_iterations=PAGERANK_MAX_ITERATIONS,
        interpolation_alpha=INTERPOLATION_ALPHA, final_score_mode=FINAL_SCORE_MODE,
        seed=SEED, probability_mode="full_vocabulary_next_token_softmax",
        pairwise_directionality="bidirectional_A_B_and_B_A",
        run_signature=experiment_signature(), python_version=platform.python_version(),
        torch_version=torch.__version__, transformers_version=transformers.__version__,
        gpu=torch.cuda.get_device_name(0), bf16_supported=torch.cuda.is_bf16_supported(),
        system_prompt=SYSTEM_PROMPT, source_csv=str(RANKING_PATH), source_parquet=str(DATASET_PATH)
    )
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def save_outputs(reranked: pd.DataFrame, pairwise: pd.DataFrame, rounds: pd.DataFrame) -> None:
    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
    eval_ranking = create_evaluation_ranking(reranked)
    files = {
        "rankings": EXPERIMENT_DIR / "rankings_top20.csv",
        "detailed": EXPERIMENT_DIR / "detailed_top20.csv",
        "pairwise": EXPERIMENT_DIR / "pairwise_comparisons.csv",
        "rounds": EXPERIMENT_DIR / "round_trace.csv",
        "config": EXPERIMENT_DIR / "config.json",
    }
    eval_ranking.to_csv(files["rankings"], index=False, encoding="utf-8-sig")
    reranked.to_csv(files["detailed"], index=False, encoding="utf-8-sig")
    if not pairwise.empty:
        pairwise.to_csv(files["pairwise"], index=False, encoding="utf-8-sig")
    if not rounds.empty:
        rounds.to_csv(files["rounds"], index=False, encoding="utf-8-sig")
    save_run_configuration(files["config"])
    # Same folder/CSV schema as existing 100Q neural rerankers:
    output_dir = PROJECT_ROOT / "reranking" / "outputs_100q_jina_vs_hybrid" / MODEL_SHORT_NAME
    output_dir.mkdir(parents=True, exist_ok=True)
    compatible_path = output_dir / f"{RETRIEVER_NAME}__{MODEL_SHORT_NAME}_rankings.csv"
    eval_ranking.to_csv(compatible_path, index=False, encoding="utf-8-sig")
    assert len(eval_ranking) == EXPECTED_QUERIES * TOP_K
    print("SAVED COMPACT EVALUATION RANKINGS:", compatible_path)
    print("SAVED DETAILED RESULTS:", files["detailed"])
    print("CACHE FOR RESUME:", EXPERIMENT_DIR / f"cache_{experiment_signature()}.jsonl")



## 15. Optional one-pair sanity test

This is a very cheap test before running a complete query.  
It verifies tokenization, chat-template rendering, GPU forward pass and A/B probabilities.


In [15]:

first_query = (
    retrieval_top20[
        retrieval_top20["query_id"]
        == retrieval_top20["query_id"].iloc[0]
    ]
    .sort_values("rank")
    .reset_index(drop=True)
)

test_query_id = str(
    first_query["query_id"].iloc[0]
)
test_query = str(
    first_query["query"].iloc[0]
)

test_cache = PairwiseCache(
    EXPERIMENT_DIR
    / f"cache_{experiment_signature()}.jsonl"
)

test_result = compare_pair_bidirectionally(
    model=model,
    tokenizer=tokenizer,
    cache=test_cache,
    query_id=test_query_id,
    query=test_query,
    candidate_i=first_query.iloc[0],
    candidate_j=first_query.iloc[1],
)

print(test_result)


PairwiseResult(query_id='Q001', candidate_first='14164', candidate_second='14331', dataset_index_first=14164, dataset_index_second=14331, p_first_when_first=0.14804424345493317, p_second_when_first=0.43779149651527405, forward_p_a=0.14804424345493317, forward_p_b=0.851935863494873, reverse_p_a=0.43779149651527405, reverse_p_b=0.5621353983879089, forward_prompt_tokens=1585, reverse_prompt_tokens=1585, first_original_tokens=298, second_original_tokens=886, first_used_tokens_forward=298, second_used_tokens_forward=886, second_used_tokens_reverse=886, first_used_tokens_reverse=298, forward_truncated=False, reverse_truncated=False, forward_winner='14331', reverse_winner='14164', order_consistent=False)



## 16. Optional full **one-query** smoke test

This runs the actual **20-candidate / 10-round PRP-Graph** pipeline for one query.

With 20 candidates, one round can compare up to 10 unique pairs.  
At 10 rounds this is about **100 pair comparisons/query**; because each comparison is run in both A/B orders, that is about **500 model forward passes/query** before cache reuse.


In [16]:

smoke_start = time.perf_counter()

smoke_reranked, smoke_pairs, smoke_rounds = (
    rerank_system(
        retrieval_top20,
        max_queries=1,
    )
)

smoke_query_id = str(
    smoke_reranked[
        "query_id"
    ].iloc[0]
)

validate_full_top20(
    retrieval_top20[
        retrieval_top20["query_id"]
        == smoke_query_id
    ].copy(),
    smoke_reranked,
)

print(
    "Smoke-test minutes:",
    round(
        (
            time.perf_counter()
            - smoke_start
        )
        / 60,
        2,
    ),
)

display(
    smoke_reranked[
        [
            "query_id",
            "candidate_id",
            "rank",
            "initial_rank",
            "rank_change",
            "prp_graph_score",
        ]
    ].head(50)
)


Loaded 1 cached pairwise comparisons.


Ministral-3-3B full Top-20 PRP-Graph:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-18 19:49:05,049 | INFO | Starting query Q001 with 20 candidates.
2026-10-18 19:49:06,298 | INFO | Query Q001 | round 1/10 | pairs=10 | cumulative=10
2026-10-18 19:49:07,654 | INFO | Query Q001 | round 2/10 | pairs=10 | cumulative=20
2026-10-18 19:49:08,992 | INFO | Query Q001 | round 3/10 | pairs=10 | cumulative=30
2026-10-18 19:49:10,221 | INFO | Query Q001 | round 4/10 | pairs=9 | cumulative=39
2026-10-18 19:49:11,429 | INFO | Query Q001 | round 5/10 | pairs=9 | cumulative=48
2026-10-18 19:49:12,678 | INFO | Query Q001 | round 6/10 | pairs=9 | cumulative=57
2026-10-18 19:49:13,910 | INFO | Query Q001 | round 7/10 | pairs=9 | cumulative=66
2026-10-18 19:49:15,054 | INFO | Query Q001 | round 8/10 | pairs=8 | cumulative=74
2026-10-18 19:49:16,175 | INFO | Query Q001 | round 9/10 | pairs=8 | cumulative=82
2026-10-18 19:49:17,281 | INFO | Query Q001 | round 10/10 | pairs=8 | cumulative=90
2026-10-18 19:49:17,301 | INFO | Completed query Q001 | pairs=90 | top=14133


VALIDATION PASSED
✓ every query has exactly 20 candidates
✓ final ranks are exactly 1..20
✓ the same original Top-20 candidate set is preserved
✓ ALL 20 candidates were eligible to move
✓ queries whose final order changed: 1 / 1
Smoke-test minutes: 0.21


,query_id,candidate_id,rank,initial_rank,rank_change,prp_graph_score
0,Q001,14133,1,6,5,0.101521
1,Q001,14331,2,2,0,0.100584
2,Q001,14332,3,7,4,0.094064
3,Q001,14181,4,10,6,0.090472
4,Q001,14190,5,11,6,0.066751
5,Q001,17296,6,5,-1,0.062690
6,Q001,14165,7,8,1,0.057925
7,Q001,14106,8,15,7,0.055627
8,Q001,14182,9,3,-6,0.054784
9,Q001,14136,10,18,8,0.051881



## 17. Full experiment — all queries

Run this after the sanity/smoke tests pass.

The JSONL cache is reused automatically, so comparisons already completed during the smoke test are not recomputed.


In [17]:

full_start = time.perf_counter()

reranked_top20, pairwise_df, round_trace_df = (
    rerank_system(
        retrieval_top20,
        max_queries=None,
    )
)

validate_full_top20(
    retrieval_top20,
    reranked_top20,
)

save_outputs(
    reranked_top20,
    pairwise_df,
    round_trace_df,
)

elapsed_minutes = (
    time.perf_counter()
    - full_start
) / 60

print()
print("=" * 88)
print(
    "MINISTRAL-3-3B FULL TOP-20 "
    "PRP-GRAPH EXPERIMENT COMPLETE"
)
print("=" * 88)
print(
    "Queries       :",
    reranked_top20[
        "query_id"
    ].nunique(),
)
print(
    "Rows          :",
    len(reranked_top20),
)
print(
    "Rows/query    :",
    reranked_top20.groupby(
        "query_id"
    ).size().unique(),
)
print(
    "Elapsed (min) :",
    round(elapsed_minutes, 2),
)
print(
    "Output dir    :",
    EXPERIMENT_DIR,
)


Loaded 90 cached pairwise comparisons.


Ministral-3-3B full Top-20 PRP-Graph:   0%|          | 0/100 [00:00<?, ?it/s]

2026-10-18 19:51:25,925 | INFO | Starting query Q001 with 20 candidates.
2026-10-18 19:51:25,931 | INFO | Query Q001 | round 1/10 | pairs=10 | cumulative=10
2026-10-18 19:51:25,932 | INFO | Query Q001 | round 2/10 | pairs=10 | cumulative=20
2026-10-18 19:51:25,933 | INFO | Query Q001 | round 3/10 | pairs=10 | cumulative=30
2026-10-18 19:51:25,934 | INFO | Query Q001 | round 4/10 | pairs=9 | cumulative=39
2026-10-18 19:51:25,935 | INFO | Query Q001 | round 5/10 | pairs=9 | cumulative=48
2026-10-18 19:51:25,936 | INFO | Query Q001 | round 6/10 | pairs=9 | cumulative=57
2026-10-18 19:51:25,937 | INFO | Query Q001 | round 7/10 | pairs=9 | cumulative=66
2026-10-18 19:51:25,937 | INFO | Query Q001 | round 8/10 | pairs=8 | cumulative=74
2026-10-18 19:51:25,938 | INFO | Query Q001 | round 9/10 | pairs=8 | cumulative=82
2026-10-18 19:51:25,940 | INFO | Query Q001 | round 10/10 | pairs=8 | cumulative=90
2026-10-18 19:51:25,954 | INFO | Completed query Q001 | pairs=90 | top=14133
2026-10-18 19:51

VALIDATION PASSED
✓ every query has exactly 20 candidates
✓ final ranks are exactly 1..20
✓ the same original Top-20 candidate set is preserved
✓ ALL 20 candidates were eligible to move
✓ queries whose final order changed: 100 / 100
SAVED COMPACT EVALUATION RANKINGS: /content/drive/MyDrive/SLM_RERANKING_100Q/reranking/outputs_100q_jina_vs_hybrid/phi_4_mini_instruct/jina_v5__phi_4_mini_instruct_rankings.csv
SAVED DETAILED RESULTS: /content/drive/MyDrive/SLM_RERANKING_100Q/outputs/phi_4_mini_instruct_full_top20_prp_graph_r10_ctx15360/jina_v5/detailed_top20.csv
CACHE FOR RESUME: /content/drive/MyDrive/SLM_RERANKING_100Q/outputs/phi_4_mini_instruct_full_top20_prp_graph_r10_ctx15360/jina_v5/cache_ee2ec853e683.jsonl

MINISTRAL-3-3B FULL TOP-20 PRP-GRAPH EXPERIMENT COMPLETE
Queries       : 100
Rows          : 2000
Rows/query    : [20]
Elapsed (min) : 22.49
Output dir    : /content/drive/MyDrive/SLM_RERANKING_100Q/outputs/phi_4_mini_instruct_full_top20_prp_graph_r10_ctx15360/jina_v5


## 18. Diagnostics

In [18]:

if not pairwise_df.empty:
    print(
        "Pairwise records          :",
        len(pairwise_df),
    )
    print(
        "Order-consistent rate     :",
        round(
            pairwise_df[
                "order_consistent"
            ].mean(),
            4,
        ),
    )
    print(
        "Forward truncation rate   :",
        round(
            pairwise_df[
                "forward_truncated"
            ].mean(),
            4,
        ),
    )
    print(
        "Reverse truncation rate   :",
        round(
            pairwise_df[
                "reverse_truncated"
            ].mean(),
            4,
        ),
    )
    print(
        "Mean forward prompt tokens:",
        round(
            pairwise_df[
                "forward_prompt_tokens"
            ].mean(),
            1,
        ),
    )

print()
print(
    "Mean absolute rank change:",
    round(
        reranked_top20[
            "rank_change"
        ].abs().mean(),
        3,
    ),
)
print(
    "Largest upward movement  :",
    int(
        reranked_top20[
            "rank_change"
        ].max()
    ),
)
print(
    "Largest downward movement:",
    int(
        reranked_top20[
            "rank_change"
        ].min()
    ),
)

display(
    reranked_top20[
        [
            "query_id",
            "candidate_id",
            "rank",
            "initial_rank",
            "rank_change",
            "prp_graph_score",
            "initial_rrf_score",
        ]
    ].head(50)
)


Pairwise records          : 9085
Order-consistent rate     : 0.2197
Forward truncation rate   : 0.0
Reverse truncation rate   : 0.0
Mean forward prompt tokens: 1333.4

Mean absolute rank change: 5.301
Largest upward movement  : 18
Largest downward movement: -19


,query_id,candidate_id,rank,initial_rank,rank_change,prp_graph_score,initial_rrf_score
0,Q001,14133,1,6,5,0.101521,0.632456
1,Q001,14331,2,2,0,0.100584,0.675110
2,Q001,14332,3,7,4,0.094064,0.628681
3,Q001,14181,4,10,6,0.090472,0.619926
4,Q001,14190,5,11,6,0.066751,0.619451
5,Q001,17296,6,5,-1,0.062690,0.633987
6,Q001,14165,7,8,1,0.057925,0.622589
7,Q001,14106,8,15,7,0.055627,0.610751
8,Q001,14182,9,3,-6,0.054784,0.662867
9,Q001,14136,10,18,8,0.051881,0.607707



## Output files

```text
outputs/
└── ministral_3_3b_instruct_2512_full_top20_prp_graph_r10_ctx15360/
    ├── logs/
    └── hybrid_bm25_jina_rrf/
        ├── rankings_top20.csv
        ├── detailed_top20.csv
        ├── pairwise_comparisons.csv
        ├── round_trace.csv
        ├── config.json
        └── cache_<signature>.jsonl
```

### Evaluation file

`rankings_top20.csv` contains:

- `system`
- `query_id`
- `category`
- `query`
- `candidate_id`
- `dataset_index`
- `rank`
- `score`

Here `rank` is the **new PRP-Graph rank 1–50** and `score` is the **actual final graph score** (or interpolated score only if you explicitly change `FINAL_SCORE_MODE`).

The full candidate set is unchanged, but every candidate can move anywhere within ranks 1–20.
